# Cell Type Classification from Single-Cell RNA-seq Data
### KNN and Decision Trees

---

**Learning Objectives**

By the end of this notebook, you will be able to:
1. Load and explore a real single-cell RNA-seq (scRNA-seq) dataset
2. Prepare features for supervised classification
3. Train and evaluate a **K-Nearest Neighbors (KNN)** classifier
4. Train and evaluate a **Decision Tree** classifier
5. Compare model performance and interpret results biologically

---

**Biological Background**

Single-cell RNA sequencing (scRNA-seq) measures gene expression across thousands of individual cells simultaneously. Each cell can be represented as a high-dimensional vector of gene expression values. A fundamental task in scRNA-seq analysis is **cell type annotation**: given a cell's expression profile, what type of cell is it?

This is a classic **multi-class classification** problem — exactly what supervised ML is designed for.

We will use the **Tabula Sapiens - Mammary dataset**, down-sampled and preprocessed to 14,922 cells × 30,497 genes.

---
## Part 0: Setup and Imports

In [ ]:
# Standard scientific Python stack
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Single-cell analysis
import scanpy as sc
import anndata as ad

# Machine learning
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.preprocessing import StandardScaler, LabelEncoder, FunctionTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, f1_score
)
from sklearn.decomposition import PCA


# Reproducibility
RANDOM_STATE = 0
np.random.seed(RANDOM_STATE)

# Plot aesthetics
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

print("All imports successful!")
print(f"scikit-learn version: {__import__('sklearn').__version__}")
print(f"scanpy version: {sc.__version__}")

---
## Part 1: Load the processed data

Scanpy uses the **AnnData** format (Annotated Data), which stores:
- `adata.X` — the expression matrix (cells × genes)
- `adata.obs` — cell-level metadata (e.g., cell type labels)
- `adata.var` — gene-level metadata
- `adata.obsm` — reduced-dimension embeddings (PCA, UMAP, etc.)

In [ ]:
# Load the dataset we saved in the preprocessing and exploration notebook
adata = sc.read_h5ad("tabula_sapiens_breast_processed.h5ad")
adata

In [ ]:
# Inspect cell-level metadata
adata.obs.head()

Next, we define the prediction target. Our goal is to predict `cell_type` from the expression profile, using cells from one donor for training and cells from the other as the test holdout.

In [ ]:
# Get the name of the prediction target: cell type labels from expert annotation ("cell_type")
target = "cell_type" 
 
print("\nCell type distribution:")
print(adata.obs[target].value_counts())
print(f"\nNumber of cell types: {adata.obs[target].nunique()}")

In [ ]:
# Visualise the class imbalance
fig, ax = plt.subplots(figsize=(10, 4))
counts = adata.obs[target].value_counts()
bars = ax.barh(counts.index, counts.values, color=sns.color_palette('tab10', len(counts)))
ax.set_xlabel('Number of cells')
ax.set_title('Cell type composition of Tabula Sapiens Mammary dataset')
for bar, val in zip(bars, counts.values):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2, str(val),
            va='center', fontsize=9)
plt.tight_layout()
plt.show()

# Discussion prompt:
# Q: The dataset is imbalanced (e.g., 4142 HR+ luminal cells vs 49 granulocytes).
#    Why might this be a problem? How does it reflect real biology?

In [ ]:
# UMAP visualisation: how cells cluster in 2D reduced space
sc.pl.umap(adata, color=["cell_type"], palette=None, size=8)

# Discussion prompt:
# Q: Which cell types appear well-separated? Which overlap?
#    How do you expect this to affect classification accuracy?

In [ ]:
# UMAP visualisation: how cells cluster in 2D reduced space
sc.pl.umap(adata, color=["donor_id"], palette=None, size=8)

# Discussion prompt:
# Q: Are cells from the two donors well integrated? Are some donor-specific?
#    How do you expect this to affect classification accuracy?

---
## Part 2: Feature Engineering and the Data Leakage Problem

### What features should we use?

We have several options:
- **Raw gene expression** (`adata.layers["counts"]`): 30,497 features — high-dimensional, potentially noisy
- **PCA components** (`adata.obsm['X_pca']`): 50 features — compressed, noise-reduced, excellent for distance-based methods
- **UMAP coordinates** (`adata.obsm['X_umap']`): 2 features — useful for visualization, but too lossy for prediction

We will use **PCA** as our primary feature space. But there is a critical methodological pitfall to address first.

---

### ⚠️ Data Leakage via Preprocessing

The dataset contains PCA coordinates in `adata.obsm['X_pca']`. These were fit on **the full dataset** — including whatever we would later call our test set. If we use those coordinates directly, the test cells have already influenced the PCA rotation, which is a form of **data leakage**.

**Why does this matter?**
- PCA finds axes of maximum variance. When test cells contribute, the principal components are subtly tuned to the entire dataset, not just the training data.
- The same rule of thumb applies when selecting highly variable genes - if the test set is included in this step, it can cause severe inflation of apparent accuracy.
- The test set is supposed to simulate completely unseen data. Leaking it into preprocessing violates this assumption.

**The correct pipeline:**
```
Raw data
    │
    ├── Train set ──► fit HVGs ──► fit Scaler ──► clip genes ──► fit PCA ──► transform ──► fit classifier
    │                     └──────────────┘────────────┘─────────────┘
    │                          (learned from training data only)
    │
    └── Test set  ──► transform only ──────────────────────────────────────────────────────► evaluate
                                     (apply the train-fit HVGs, scaler & PCA, never refit)
```

sklearn's `Pipeline` enforces this automatically: during cross-validation it re-fits HVGs, the scaler and PCA **inside each fold**, so no fold's validation data ever leaks into its own preprocessing. Note that we've incorporated the clipping step in the pipeline as well, however it doesn't learn anything from the training data.

In [ ]:
# We start from log-normalised gene expression, which is already saved in the adata.X slot from our preprocessing notebook
# Total normalization and logarithmization depend only on each cell's own values, and do not leak data 
# PCA will be fit exclusively on training data inside a Pipeline.
X_raw = adata.X                           
y_raw = adata.obs[target].values    

print(f"Raw expression matrix shape: {X_raw.shape}")
print(f"Number of labels: {len(y_raw)}")
print(y_raw.categories)


In [ ]:
# Encode string labels as integers for sklearn
le = LabelEncoder()
y = le.fit_transform(y_raw)

# Inspect the mapping
label_map = dict(zip(le.transform(le.classes_), le.classes_))
print("Label encoding:")
for code, name in sorted(label_map.items()):
    print(f"  {code} → {name}")

In [ ]:
# We want to train on data from one donor, and test on the other
# Split BEFORE the preprocessing Pipeline — the test set must be truly held out

train_mask = (adata.obs["donor_id"] == "TSP27").values
X_train = X_raw[train_mask].toarray()
y_train = y[train_mask]

test_mask = (adata.obs["donor_id"] == "TSP4").values
X_test = X_raw[test_mask].toarray()
y_test = y[test_mask]

print(f"Training set: {X_train.shape[0]} cells × {X_train.shape[1]} genes")
print(f"Test set:     {X_test.shape[0]} cells × {X_test.shape[1]} genes")
print(f"\nClass distribution in training set:")
for code, name in sorted(label_map.items()):
    n = np.sum(y_train == code)
    print(f"  {name}: {n}")

In [ ]:
# ---------------------------------------------------------------------------
# Build a preprocessing pipeline: HighlyVariableGenes → StandardScaler → Clip outliers → PCA
# This object will be fit ONLY on X_train, then applied to X_test.
# When used inside cross_val_score, sklearn re-fits it fresh in every fold.
# ---------------------------------------------------------------------------

# We want to reduce the dimensionality of the dataset and include only the most informative genes
# This step is known as feature selection
# The following wrapper for scanpy's pp.highly_variable_genes allows us to incorporate this step into a pipeline
class HighlyVariableGenes(BaseEstimator, TransformerMixin):
    """
    Select the top-N genes by normalized dispersion, computed on training
    cells only.  Expects logarithmized data.

    fit()       — learns which genes to keep (from training data only)
    transform() — applies that fixed gene mask to any split
    """

    def __init__(self, n_top_genes=2000, flavor='seurat', batch_key=None):
        self.n_top_genes = n_top_genes
        self.flavor      = flavor
        self.batch_key   = batch_key

    def fit(self, X, y=None):
        # Build a minimal AnnData from the training matrix
        adata_tmp = ad.AnnData(X)

        # Run scanpy's hvg implementation
        sc.pp.highly_variable_genes(
            adata_tmp,
            n_top_genes=self.n_top_genes,
            flavor=self.flavor,
            batch_key=self.batch_key,
        )

        # Store the boolean mask — this is what transform() will apply
        self.selected_mask_ = adata_tmp.var['highly_variable'].values
        self.n_genes_in_    = X.shape[1]
        return self

    def transform(self, X, y=None):
        if sp.issparse(X):
            # Column slicing on CSR is slow; convert to CSC first
            return X.tocsc()[:, self.selected_mask_].tocsr()
        return X[:, self.selected_mask_]

    def get_feature_names_out(self, gene_names=None):
        """Return selected gene names if the original list is provided."""
        if gene_names is not None:
            return np.array(gene_names)[self.selected_mask_]
        return np.where(self.selected_mask_)[0]

    
N_COMPONENTS = 50 # number of PCA components to use
preprocessor = Pipeline([
    ('hvg',     HighlyVariableGenes(n_top_genes=2000)),
    ('scaler', StandardScaler()),
    ("clip",    FunctionTransformer(np.clip, kw_args={'a_min': -10, 'a_max': 10})),
    ('pca',    PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE))
])

# Fit on training data only, then transform both sets
X_train_pca = preprocessor.fit_transform(X_train)   # fit + transform
X_test_pca  = preprocessor.transform(X_test)        # transform ONLY — no refitting!

pca_obj = preprocessor.named_steps['pca']
print(f"\nPCA fit on training cells only ({X_train.shape[0]} cells).")
print(f"Variance explained by {N_COMPONENTS} PCs: "
      f"{pca_obj.explained_variance_ratio_.sum()*100:.1f}%")
print(f"\nTransformed shapes — train: {X_train_pca.shape}, test: {X_test_pca.shape}")

---
## Part 3: Logistic Regression Classifier

### How Logistic Regression Works

Despite the name, logistic regression is a **classification** algorithm.
For each class k it learns a weight vector w_k and bias b_k, then
computes a score for every cell:

    score_k(x) = w_k · x + b_k

Scores are converted to probabilities via the sigmoid function (for binary classification) or 
the softmax function (for multiclass problems). The predicted class is the one with the highest probability.

**Key hyperparameter:** C (inverse regularisation strength)
- Small C → strong regularisation → simpler boundary (high bias)
- Large C → weak regularisation  → complex boundary (high variance)

**Advantages:**
- Fast at prediction time (a dot product per class)
- Well-calibrated probabilities via softmax
- Coefficients are interpretable
- Handles high-dimensional data well when regularised

In [ ]:
# Hyperparameter tuning: regularisation strength C
# IMPORTANT: We pass X_train and let the Pipeline re-fit the hvg+scaler+PCA
# inside every CV fold. This is the leak-free way to do cross-validation.

C_values = np.logspace(-3, 3, 13)   # 0.001 → 1000, 13 points
cv_scores = []

# We are using F1 score to evaluate the model. Macro F1 computes F1 independently 
# for each class and averages them equally, so rare classes contribute as much as 
# common ones to the overall score.
metric = 'f1_macro'

# Split the training set into train and validation splits, preserving the frequency
# of classes in each split to ensure cross-validation is performed on all classes
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for C in C_values:
    # Full pipeline: preprocessing (fit inside fold) + classifier
    pipe_lr = Pipeline([
        ('hvg',     HighlyVariableGenes(n_top_genes=2000)),
        ('scaler',  StandardScaler()),
        ("clip",    FunctionTransformer(np.clip, kw_args={'a_min': -10, 'a_max': 10})),
        ('pca',     PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)),
        ('lr',      LogisticRegression(C=C,
                                       solver='lbfgs',
                                       max_iter=1000,
                                       random_state=RANDOM_STATE)),
    ])

    # cross_val_score calls pipe_lr.fit() on each training fold — PCA is re-fit each time
    scores = cross_val_score(pipe_lr, X_train, y_train, cv=cv, scoring=metric)
    cv_scores.append(scores.mean())

best_C = C_values[np.argmax(cv_scores)]
print(f"Best C by 5-fold CV: {best_C:.4f} "
      f"(CV {metric}: {max(cv_scores):.3f})")

In [ ]:
# Hyperparameter tuning: finding the optimal C
cv_loss_lr = [1 - f for f in cv_scores]

fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogx(C_values, cv_loss_lr, 's-', color='steelblue',
            linewidth=2, label='CV loss (1 - macro F1)')
ax.axvline(best_C, color='green', linestyle='--',
           label=f'Best C={best_C:.4f}')
ax.set_xlabel('C (inverse regularisation strength, log scale)')
ax.set_ylabel('Macro F1 Loss (1 − F1)')
ax.set_title('Logistic Regression: Hyperparameter Tuning (C selection)')
ax.legend()
plt.tight_layout()
plt.show()

# Discussion prompts:
# Q: Why is C searched on a log scale rather than a linear scale?
# Q: Why must the Pipeline include the scaler and PCA, not just the KNN step?

In [ ]:
# Retrain with best C and evaluate
lr_best = LogisticRegression(
    C=best_C,
    solver='lbfgs',
    max_iter=1000,
    random_state=RANDOM_STATE
)
lr_best.fit(X_train_pca, y_train)
y_pred_lr_best = lr_best.predict(X_test_pca)

acc_lr_best = accuracy_score(y_test, y_pred_lr_best)
print(f"Logistic Regression (C={best_C}) Test Accuracy: {acc_lr_best:.3f}")

f1_lr_best  = f1_score(y_test, y_pred_lr_best, average='macro', zero_division=0)
print(f"Logistic Regression (C={best_C}) Test F1 Macro: {f1_lr_best:.3f}")


In [ ]:
# Per-class performance
print(f"Classification Report — LR (C={best_C}):\n")
print(classification_report(
    y_test, y_pred_lr_best,
    target_names=le.classes_,
    zero_division=0
))

# Discussion prompt:
# Q: Which cell types have the highest precision/recall? Why might that be?
# Q: What does 'support' tell you? How does class size affect performance?

In [ ]:
# Confusion matrix
cm_lr = confusion_matrix(y_test, y_pred_lr_best, normalize='true')
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay(confusion_matrix=cm_lr,
                       display_labels=le.classes_).plot(
    ax=ax, xticks_rotation=90, colorbar=True, cmap='Blues', values_format='.2f'
)
ax.set_title(f'Confusion Matrix — Logistic Regression (C={best_C:.4f})',
             fontsize=13)
plt.tight_layout()
plt.grid(False)
plt.show()

# Discussion prompt:
# Q: Which cell types are most often confused with each other?
# Q: Does this make biological sense? (e.g., cell subtypes may be similar)

In [ ]:
# Coefficient heatmap
# Large positive values mean that PC pushes strongly toward that class;
# large negative values push away from it.

coef_df = pd.DataFrame(
    lr_best.coef_,
    index=le.classes_,
    columns=[f'PC{i+1}' for i in range(N_COMPONENTS)]
)

# Show only the top 15 PCs by maximum absolute weight across all classes
top_pcs = coef_df.abs().max(axis=0).nlargest(15).index

fig, ax = plt.subplots(figsize=(13, 6))
sns.heatmap(
    coef_df[top_pcs],
    cmap='RdBu_r',
    center=0,
    linewidths=0.4,
    ax=ax,
    yticklabels=True,
    cbar_kws={'label': 'Coefficient weight'}
)
ax.set_title('Logistic Regression coefficients\n'
             '(top 15 PCs by max absolute weight across classes)',
             fontsize=12)
ax.set_xlabel('Principal Component')
ax.set_ylabel('Cell type')
plt.tight_layout()
plt.show()

# Discussion prompts:
# Q: Which PCs most strongly distinguish HR+ from secretory luminal cells?

---
## Part 4: K-Nearest Neighbors (KNN) Classifier

### How KNN Works

KNN is a **non-parametric, instance-based** algorithm. It makes no assumptions about data distribution.

**Prediction for a new cell:**
1. Compute the distance (usually Euclidean) between the new cell and all training cells
2. Find the **K nearest neighbours**
3. Assign the **majority class** among those K neighbours

```
         [?] ← new cell
        / | \
       B  B  T       K=3: 2 B-cells, 1 T-cell → predict B-cell
```

**Key hyperparameter:** `K` (number of neighbours)
- Small K → complex boundary, low bias, high variance (overfitting risk)
- Large K → smooth boundary, high bias, low variance (underfitting risk)

**Advantages**
- Makes no assumptions about decision boundary shape (can capture complex, non-linear boundaries between classes)
- Naturally multiclass
- Interpretable per prediction (you can inspect which specific reference cells drove a classification decision)

In [ ]:
# Hyperparameter tuning: finding the optimal K

k_values = list(range(1, 15))
cv_scores = []
metric = 'f1_macro'

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for k in k_values:
    # Full pipeline: preprocessing (fit inside fold) + classifier
    pipe_k = Pipeline([
        ('hvg',     HighlyVariableGenes(n_top_genes=2000)),
        ('scaler', StandardScaler()),
        ("clip",    FunctionTransformer(np.clip, kw_args={'a_min': -10, 'a_max': 10})),
        ('pca',    PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)),
        ('knn',    KNeighborsClassifier(n_neighbors=k, metric='euclidean', weights='distance'))
    ])
    # cross_val_score calls pipe_k.fit() on each training fold — PCA is re-fit each time
    scores = cross_val_score(pipe_k, X_train, y_train, cv=cv, scoring=metric)
    cv_scores.append(scores.mean())

best_k = k_values[np.argmax(cv_scores)]
print(f"Best K by 5-fold CV: K={best_k} (CV {metric}: {max(cv_scores):.3f})")

# Plot K vs score
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(k_values, cv_scores, 'o-', color='steelblue', linewidth=2, markersize=5)
ax.axvline(best_k, color='tomato', linestyle='--', label=f'Best K={best_k}')
ax.set_xlabel('Number of Neighbours (K)')
ax.set_ylabel(f'5-Fold CV {metric}')
ax.set_title('KNN: Hyperparameter Tuning (K selection)')
ax.legend()
plt.tight_layout()
plt.show()

# Discussion prompt:
# Q: What happens to the score at K=1? What does this tell us about overfitting?

In [ ]:
# Retrain with best K and evaluate on held-out test set.
# X_train_pca / X_test_pca were produced by the pipeline fit on training data only.
knn_best = KNeighborsClassifier(n_neighbors=best_k, metric='euclidean')
knn_best.fit(X_train_pca, y_train)
y_pred_knn_best = knn_best.predict(X_test_pca)

acc_knn_best = accuracy_score(y_test, y_pred_knn_best)
print(f"KNN (K={best_k}) Test Accuracy: {acc_knn_best:.3f}")

f1_knn_best = f1_score(y_test, y_pred_knn_best, average="macro")
print(f"KNN (K={best_k}) Test F1 Macro: {f1_knn_best:.3f}")

# Per-class performance
print(f"Classification Report — KNN (K={best_k}):\n")
print(classification_report(
    y_test, y_pred_knn_best,
    target_names=le.classes_,
    zero_division=0
))

---
## Part 5: Decision Tree Classifier

### How Decision Trees Work

A Decision Tree recursively **partitions the feature space** by asking binary questions:

```
         PC1 > 2.3?
        /           \
       Yes           No
      /               \
   PC3 > -1.0?      CD14+ Monocyte
    /       \
 B cell    T cell
```

At each node, the algorithm selects the split that **maximally reduces impurity** (using Gini impurity or entropy).

$$\text{Gini}(p) = 1 - \sum_k p_k^2$$

**Key hyperparameters:**
- `max_depth`: limits how deep the tree can grow (regularization)
- `min_samples_split`: minimum samples required to split a node
- `criterion`: 'gini' or 'entropy'

**Advantages:**
- Interpretable (you can visualize the tree)
- Fast at prediction time
- Handles different feature scales naturally

In [ ]:
# Hyperparameter tuning: optimal max_depth
depths = list(range(1, 21))
cv_scores, train_scores = [], []
metric = 'f1_macro'

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for d in depths:
    pipe_dt = Pipeline([
            ('hvg',     HighlyVariableGenes(n_top_genes=2000)),
            ('scaler', StandardScaler()),
            ("clip",    FunctionTransformer(np.clip, kw_args={'a_min': -10, 'a_max': 10})),
            ('pca',    PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)),
            ("dt", DecisionTreeClassifier(criterion="gini", max_depth=d, random_state=RANDOM_STATE))
        ])

    # CV F1 — PCA and all preprocessing re-fit inside each fold
    scores = cross_val_score(pipe_dt, X_train, y_train, cv=cv, scoring=metric)
    cv_scores.append(scores.mean())

    # Train on the full training set to get optimistic in-sample score for bias-variance plot
    dt = DecisionTreeClassifier(criterion='gini', max_depth=d, random_state=RANDOM_STATE)
    # Training score on PCA-transformed train set
    dt.fit(X_train_pca, y_train)
    train_scores.append(f1_score(y_train, dt.predict(X_train_pca), average="macro"))

best_depth = depths[np.argmax(cv_scores)]
print(f"Best max_depth by 5-fold CV: {best_depth} "
      f"(CV {metric}: {max(cv_scores):.3f})")

# Plot depth vs score
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(depths, cv_scores, 'o-', color='steelblue', linewidth=2, markersize=5)
ax.axvline(best_depth, color='tomato', linestyle='--', label=f'Best depth={best_depth}')
ax.set_xlabel('Max depth')
ax.set_ylabel(f'5-Fold CV {metric}')
ax.set_title('Decision Tree: Hyperparameter Tuning (max depth selection)')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Convert F1 scores to loss (1 - F1)
train_loss  = [1 - f for f in train_scores]
cv_loss     = [1 - f for f in cv_scores]

print(f"Best max_depth by 5-fold CV: {best_depth} "
      f"(CV F1 macro loss: {min(cv_loss):.3f})")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(depths, train_loss, 'o-', color='tomato',
        label='Training loss (1 - macro F1)', linewidth=2)
ax.plot(depths, cv_loss,    's-', color='steelblue',
        label='CV loss (1 - macro F1)',       linewidth=2)
ax.axvline(best_depth, color='green', linestyle='--',
           label=f'Best depth={best_depth}')
ax.set_xlabel('max_depth')
ax.set_ylabel('Macro F1 Loss (1 - F1)')
ax.set_title('Decision Tree: Bias-Variance Tradeoff')
ax.legend()
plt.tight_layout()
plt.show()

# Discussion prompts:
# Q: At high depths, training loss approaches 0 but CV loss plateaus. What is this called?
# Q: Does the optimal depth change compared to when we use accuracy as the CV metric?
#    If so, why might f1_macro prefer a different depth?


In [ ]:
# Retrain with best depth and evaluate
dt_best = DecisionTreeClassifier(criterion='gini', max_depth=best_depth, random_state=RANDOM_STATE)
dt_best.fit(X_train_pca, y_train)
y_pred_dt_best = dt_best.predict(X_test_pca)

acc_dt_best = accuracy_score(y_test, y_pred_dt_best)
print(f"Decision Tree (max_depth={best_depth}) Test Accuracy: {acc_dt_best:.3f}")

f1_dt_best = f1_score(y_test, y_pred_dt_best, average="macro")
print(f"Decision Tree (max_depth={best_depth}) Test F1 Macro: {f1_dt_best:.3f}")

print("\nClassification Report — Decision Tree:\n")
print(classification_report(
    y_test, y_pred_dt_best,
    target_names=le.classes_,
    zero_division=0
))

In [ ]:
# Which PCs are most used in splitting decisions?
# Feature names: PC1, PC2, ..., PC50
feature_names = [f'PC{i+1}' for i in range(N_COMPONENTS)]
#Feature importances
importances = dt_best.feature_importances_
top_n = 15
top_idx = np.argsort(importances)[::-1][:top_n]

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(
    range(top_n),
    importances[top_idx],
    color=sns.color_palette('viridis', top_n)
)
ax.set_xticks(range(top_n))
ax.set_xticklabels([feature_names[i] for i in top_idx], rotation=45, ha='right')
ax.set_ylabel('Gini Importance')
ax.set_title(f'Top {top_n} Feature Importances — Decision Tree (max_depth={best_depth})')
plt.tight_layout()
plt.show()

# Discussion prompt:
# Q: PC1 and PC2 capture the most variance — are they also the most discriminative?
# Q: How could you connect these PCs back to biology? (Hint: look at gene loadings)

In [ ]:
# Visualise a shallow tree (max_depth=3) for interpretability
dt_shallow = DecisionTreeClassifier(criterion='gini', max_depth=3, random_state=RANDOM_STATE)
dt_shallow.fit(X_train_pca, y_train)

fig, ax = plt.subplots(figsize=(12, 6))
plot_tree(
    dt_shallow,
    feature_names=feature_names,
    class_names=le.classes_,
    filled=True,
    rounded=True,
    fontsize=6,
    ax=ax,
    impurity=True
)
ax.set_title('Decision Tree — Cell Type Classification', fontsize=13)
plt.tight_layout()
plt.show()

# Discussion prompt:
# Q: Which PCA component is the most informative first split?
# Q: Can you interpret what that component might represent biologically?

---
## Part 6: Model Comparison

Let's put the models side by side and compare their strengths and weaknesses.

In [ ]:
# Head-to-head comparison
results = {
    'Model': [f'LR (C={best_C})', f'KNN (K={best_k})', f'Decision Tree (depth={best_depth})'],
    'Test Accuracy': [
        acc_lr_best,
        acc_knn_best,
        acc_dt_best
    ],
    'Test F1 Macro': [
        f1_lr_best,
        f1_knn_best,
        f1_dt_best
    ]
}
df_results = pd.DataFrame(results)
df_results

In [ ]:
# Side-by-side confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(15, 6), sharey=True)

for ax, y_pred, title in zip(
    axes,
    [y_pred_lr_best, y_pred_knn_best, y_pred_dt_best],
    [f'LR (C={best_C})', f'KNN (K={best_k})', f'Decision Tree (depth={best_depth})']
):
    cm = confusion_matrix(y_test, y_pred, normalize='true')
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
    disp.plot(ax=ax, xticks_rotation=90, colorbar=False, cmap='Blues', values_format='.2f')
    ax.set_title(title, fontsize=12)
    ax.grid(False)

plt.suptitle('Confusion Matrix Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Per-class F1 comparison
from sklearn.metrics import f1_score

f1_lr = f1_score(y_test, y_pred_lr_best, average=None, labels=list(range(len(le.classes_))), zero_division=0)
f1_knn = f1_score(y_test, y_pred_knn_best, average=None, labels=list(range(len(le.classes_))), zero_division=0)
f1_dt  = f1_score(y_test, y_pred_dt_best,  average=None, labels=list(range(len(le.classes_))), zero_division=0)

df_f1 = pd.DataFrame({
    'Cell Type': le.classes_,
    f'LR F1': f1_lr,
    f'KNN F1': f1_knn,
    f'DT F1':  f1_dt
})

x = np.arange(len(le.classes_))
width = 0.25   # narrower bars to fit three models

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - width, f1_lr,  width, label=f'LR (C={best_C:.3f})', color='seagreen', alpha=0.85)
ax.bar(x, f1_knn, width, label=f'KNN (K={best_k})', color='steelblue', alpha=0.85)
ax.bar(x + width, f1_dt,  width, label=f'DTree (depth={best_depth})', color='tomato', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(le.classes_, rotation=40, ha='right', fontsize=9)
ax.set_ylabel('F1 Score')
ax.set_title('Per-Class F1 Score')
ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

# Discussion prompt:
# Q: For rare cell types, which model performs better?
# Q: What strategies could help with rare cell types? (Hint: oversampling, class weights)

---
## Part 7: Decision Boundaries in 2D (UMAP Space)

To build intuition, let's visualise how KNN and Decision Tree partition the UMAP space.

> **Note:** We train on 2D UMAP coordinates here purely for visualisation. In practice, we train on the full PCA space.

In [ ]:
from umap import UMAP

# Note: we skip HVG here since we already have X_train_pca from the main
# pipeline — we can reduce those 50 PCs to 2D rather than starting from
# raw counts again, which is both faster and consistent with the main model.

umap_pipeline = Pipeline([
    ('umap', UMAP(n_components=2, random_state=RANDOM_STATE,
                  n_neighbors=15, min_dist=0.1))
])

# Fit UMAP on training PCA coords only, transform both splits
X_train_umap = umap_pipeline.fit_transform(X_train_pca)
X_test_umap  = umap_pipeline.transform(X_test_pca)

print(f"Train UMAP shape: {X_train_umap.shape}")
print(f"Test UMAP shape:  {X_test_umap.shape}")
print("UMAP fit on training cells only — test cells projected into that space")

In [ ]:
# Build meshgrid over the training UMAP space
h = 0.1
x_min, x_max = X_train_umap[:, 0].min() - 0.5, X_train_umap[:, 0].max() + 0.5
y_min, y_max = X_train_umap[:, 1].min() - 0.5, X_train_umap[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))
grid = np.c_[xx.ravel(), yy.ravel()]

n_classes  = len(le.classes_)
colors_rgb = np.array(sns.color_palette('tab10', n_classes))

# Each model is trained on X_train_umap (2D, leak-free) and evaluated on X_test_umap
models_2d = [
    (KNeighborsClassifier(n_neighbors=best_k), f'KNN (K={best_k})'),
    (DecisionTreeClassifier(max_depth=best_depth, random_state=RANDOM_STATE), f'Decision Tree (depth={best_depth})'),
]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, (model, title) in zip(axes, models_2d):
    # Fit on leak-free training UMAP coordinates
    model.fit(X_train_umap, y_train)

    # Decision regions over the meshgrid
    Z = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25,
                levels=np.arange(-0.5, n_classes), cmap='tab10')

    # Training cells
    for k, ct in enumerate(le.classes_):
        mask = y_train == k
        ax.scatter(X_train_umap[mask, 0], X_train_umap[mask, 1],
                   s=18, alpha=0.5, color=colors_rgb[k],
                   label=ct if k == 0 else '_nolegend_')

    # Test cells — correct and incorrect, distinguished by marker
    y_pred_2d = model.predict(X_test_umap)
    correct  = y_pred_2d == y_test
    for k, ct in enumerate(le.classes_):
        mask_k = y_test == k
        # Correct test predictions
        ax.scatter(X_test_umap[mask_k &  correct, 0],
                   X_test_umap[mask_k &  correct, 1],
                   s=40, color=colors_rgb[k], edgecolors='white',
                   linewidths=0.5, zorder=3, marker='o')
        # Wrong test predictions
        ax.scatter(X_test_umap[mask_k & ~correct, 0],
                   X_test_umap[mask_k & ~correct, 1],
                   s=60, color=colors_rgb[k], edgecolors='black',
                   linewidths=1.0, zorder=4, marker='X')

    acc_2d = accuracy_score(y_test, y_pred_2d)
    f1_2d  = f1_score(y_test, y_pred_2d, average='macro',
                      zero_division=0)
    ax.set_title(f'{title}\nacc={acc_2d:.2f}, macro F1={f1_2d:.2f}',
                 fontsize=11)
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')

# Shared legend for cell types
handles = [plt.scatter([], [], color=colors_rgb[k], s=30, label=ct)
           for k, ct in enumerate(le.classes_)]
# Markers legend
handles += [
    plt.scatter([], [], s=40, c='grey', edgecolors='white',
                linewidths=0.5, marker='o', label='Test — correct'),
    plt.scatter([], [], s=60, c='grey', edgecolors='black',
                linewidths=1.0, marker='X', label='Test — wrong'),
]
axes[1].legend(handles=handles, bbox_to_anchor=(1.05, 1),
               loc='upper left', fontsize=10)

plt.suptitle(
    'Decision Boundaries in UMAP Space\n'
    '(UMAP fit on training cells only; test cells projected in)',
    fontsize=13
)
plt.tight_layout()
plt.show()

# Discussion prompts:
# Q: KNN boundaries are smooth; DT boundaries are axis-aligned rectangles. Why?
# Q: Do the misclassified test cells (X markers) sit near class boundaries,
#    or are some deep inside the wrong region?

---
## Part 8: Bonus — Using Raw Gene Expression

What if we skip PCA and use raw gene expression as features? Let's compare.

In [ ]:
# Bonus: compare using raw 30497-gene expression vs. PCA features.
# We already have X_train / X_test (raw) from Part 2 — use those directly.
# No extra split needed; scaling is applied via StandardScaler so distances are fair.

scaler_raw = StandardScaler().fit(X_train)   # fit on training only
X_tr_raw_sc = scaler_raw.transform(X_train)
X_te_raw_sc = scaler_raw.transform(X_test)   # apply same scaler to test

# LR on scaled raw expression (no PCA)
lr_raw = LogisticRegression(C=best_C, solver='lbfgs', max_iter=1000,random_state=RANDOM_STATE)
lr_raw.fit(X_tr_raw_sc, y_train)
f1_lr_raw = f1_score(y_test, lr_raw.predict(X_te_raw_sc), average="macro")
acc_lr_raw = accuracy_score(y_test, lr_raw.predict(X_te_raw_sc))

# KNN on scaled raw expression (no PCA)
knn_raw = KNeighborsClassifier(n_neighbors=best_k, metric="euclidean", weights="distance")
knn_raw.fit(X_tr_raw_sc, y_train)
f1_knn_raw = f1_score(y_test, knn_raw.predict(X_te_raw_sc), average="macro")
acc_knn_raw = accuracy_score(y_test, knn_raw.predict(X_te_raw_sc))

# Decision Tree on scaled raw expression (no PCA)
dt_raw = DecisionTreeClassifier(max_depth=best_depth, random_state=RANDOM_STATE)
dt_raw.fit(X_tr_raw_sc, y_train)
f1_dt_raw = f1_score(y_test, dt_raw.predict(X_te_raw_sc), average="macro")
acc_dt_raw = accuracy_score(y_test, dt_raw.predict(X_te_raw_sc))

# Head-to-head comparison
results = {
    'Model': [f'LR (PCA 50 dims)', f'LR (Raw 30497 genes)',
              f'KNN (PCA 50 dims)', f'KNN (Raw 30497 genes)', 
              f'Decision Tree (PCA 50 dims)', f'Decision Tree (Raw 30497 genes)'],
    'Test Accuracy': [
        acc_lr_best,
        acc_lr_raw,
        acc_knn_best,
        acc_knn_raw,
        acc_dt_best,
        acc_dt_raw
    ],
    'Test F1 Macro': [
        f1_lr_best,
        f1_lr_raw,
        f1_knn_best,
        f1_knn_raw,
        f1_dt_best,
        f1_dt_raw
    ]
}
df_results = pd.DataFrame(results)
df_results

# Discussion prompt:
# Q: Does PCA hurt or help? Why?
# Q: KNN is particularly sensitive to high dimensions — what is this called?
# Q: Why does the Decision Tree perform similarly with or without PCA?


---
## Summary & Discussion

| | **Logistic Regression** | **KNN** | **Decision Tree** |
|---|---|---|---|
| **Type** | Parametric, discriminative | Instance-based, lazy | Rule-based, eager |
| **Training** | Learns weight vectors | Just stores data | Builds tree structure |
| **Prediction** | Fast (dot product per class) | Slow (compute distances) | Fast (follow rules) |
| **Interpretability** | Global (coefficients) | Per-prediction (neighbours) | Global (tree rules) |
| **Overfitting risk** | Low with regularisation | High at small K | High at large depth |
| **Handles imbalance** | Moderately | Poorly | Moderately |
| **Key hyperparameter** | C (regularisation strength) | K (neighbours) | max_depth |

### Key Takeaways

1. **Feature choice matters** — PCA-reduced features often outperform raw high-dimensional data
   for KNN due to the curse of dimensionality; logistic regression and decision trees are less sensitive to this.

2. **Preprocessing must be fit on training data only** — applying normalisation, HVG selection,
   scaling, or PCA to the full dataset before splitting leaks test information and inflates accuracy estimates.

3. **Hyperparameter tuning is essential** — Always use cross-validation with a full pipeline,
   never the test set, to select hyperparameters. The pipeline ensures preprocessing is re-fit inside every fold.

4. **Class imbalance is real** — Rare cell types are consistently harder to
   predict. Check F1 scores for each class, and consider class weights or SMOTE.

5. **No single model wins on all criteria** — KNN is interpretable per prediction, Decision Trees
   are interpretable globally, and Logistic Regression gives well-calibrated probabilities and inspectable coefficients. The right choice depends on your priorities.

### Going Further

- **Random Forests** — ensembles of many decision trees; usually outperform a single tree and provide robust feature importances
- **Support Vector Machines (SVM)** — effective for high-dimensional gene expression data with a linear kernel
- **Batch correction** — methods such as Harmony or scVI can align donor-specific effects before classification
- **scANVI / scVI** — deep learning approaches specifically designed for scRNA-seq that model the generative process of count data
- **Marker gene selection** — instead of PCA, use known biological markers as features

In [ ]:
# ============================================================
# EXERCISE: Try it yourself!
# ============================================================
#
# 1. Train a KNN classifier using UMAP coordinates (X_umap) as features.
#    Does it perform better or worse than PCA? Why?
#
# 2. Train a Decision Tree with criterion='entropy' instead of 'gini'.
#    Does it change the accuracy or the tree structure?
#
# 3. Identify the top 10 most important genes in the Decision Tree 
#    (Hint: use dt_raw.feature_importances_ and adata.var_names)
#    Do these correspond to known mammary marker genes?
#
# 4. (Advanced) Use class_weight='balanced' in DecisionTreeClassifier.
#    Does this improve performance on the rare cell types?

# Your code here:
